Text Classifer Using Machine Learning

Build and eval a model using Multinomial Naive Bayes imports cleaned text into 6 emotions. 

Feature representation: Count Vectorizer and TF IDF 

1. Does negation aware preprocessing improve classification performance? YES
2. Comparing different vectorizers to see difference in performance? 
3. Compare this model with emotions modelling Logistic Regression under identical experiemental conditions

Measuring if the emotion classifier model is good? 
1. 5 fold cross validation
2. Accuracy
3. Macro F1 = Precision and Recall 
4. Per Class performance via confusion matrix 

In [62]:

# Data handling
import ast
import pandas as pd
import numpy as np


# Machine learning: data splitting and evaluation
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

# Text feature extraction
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer
)

# Machine learning models
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import confusion_matrix

# Model performance metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Visualization
import matplotlib.pyplot as plt


Loading the Data - 

Load preprocessed emotion dataset prior to training model using clean_tokens and clean_tokens_neg

In [63]:

# Load the preprocessed emotion dataset
df = pd.read_csv("./data/emotions_clean.csv")

# Display the first five rows
display(df.head())

# Check the number of rows and columns
print("Dataset shape:", df.shape)

# Display the column names
print("Columns:", df.columns.tolist())

#print(df.info())


,Unnamed: 0,text,emotion,doc,tokens,lemmas,clean_tokens,adjectives,clean_tokens_neg,word_count
0,0,i didnt feel humiliated,sadness,i didnt feel humiliated,"['i', 'did', 'nt', 'feel', 'humiliated']","['I', 'do', 'not', 'feel', 'humiliated']","['feel', 'humiliated']",['humiliated'],"['not_feel', 'humiliated']",5
1,1,i can go from feeling so hopeless to so damned...,sadness,i can go from feeling so hopeless to so damned...,"['i', 'can', 'go', 'from', 'feeling', 'so', 'h...","['I', 'can', 'go', 'from', 'feel', 'so', 'hope...","['feel', 'hopeless', 'damned', 'hopeful', 'car...","['hopeless', 'hopeful', 'awake']","['feel', 'hopeless', 'damned', 'hopeful', 'car...",21
2,2,im grabbing a minute to post i feel greedy wrong,anger,im grabbing a minute to post i feel greedy wrong,"['i', 'm', 'grabbing', 'a', 'minute', 'to', 'p...","['I', 'm', 'grab', 'a', 'minute', 'to', 'post'...","['m', 'grab', 'minute', 'post', 'feel', 'greed...","['greedy', 'wrong']","['m', 'grab', 'minute', 'post', 'feel', 'greed...",11
3,3,i am ever feeling nostalgic about the fireplac...,love,i am ever feeling nostalgic about the fireplac...,"['i', 'am', 'ever', 'feeling', 'nostalgic', 'a...","['I', 'be', 'ever', 'feel', 'nostalgic', 'abou...","['feel', 'nostalgic', 'fireplace', 'know', 'pr...",['nostalgic'],"['feel', 'nostalgic', 'fireplace', 'know', 'pr...",18
4,4,i am feeling grouchy,anger,i am feeling grouchy,"['i', 'am', 'feeling', 'grouchy']","['I', 'be', 'feel', 'grouchy']","['feel', 'grouchy']",['grouchy'],"['feel', 'grouchy']",4


Dataset shape: (16000, 10)
Columns: ['Unnamed: 0', 'text', 'emotion', 'doc', 'tokens', 'lemmas', 'clean_tokens', 'adjectives', 'clean_tokens_neg', 'word_count']


In [64]:

# Count examples in each emotion category to identify potential class imbalances and data quality issues before model training 
print("Emotion distribution:")
print(df["emotion"].value_counts())

# Check for missing values in required columns
required_columns = [
    "emotion",
    "clean_tokens",
    "clean_tokens_neg"
]

print("\nMissing values:")
print(df[required_columns].isna().sum())

#print(df.groupby("emotion").size())


Emotion distribution:
emotion
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64

Missing values:
emotion             0
clean_tokens        0
clean_tokens_neg    0
dtype: int64


Text PreProcessing and feature preperation

clean_tokens
clean_tokens_neg

converted into space separated strings to be used with CountVectorizer and TF-IDF

In [65]:

# Convert stored token strings into Python lists
df["clean_list"] = df["clean_tokens"].apply(ast.literal_eval)
df["neg_list"] = df["clean_tokens_neg"].apply(ast.literal_eval) #Safely interpret a string containing a Python literal, such as a list.


# Convert token lists into space-separated text
#BEFORE — One string containing list syntax
#"['not_happy', 'today']"
#AFTER — An actual Python list
#["not_happy", "today"]

df["clean_text"] = df["clean_list"].apply(" ".join)
df["neg_text"] = df["neg_list"].apply(" ".join)

# Set the model input and target
X = df["neg_text"] #can change to "clean_text" to compare the effect of preprocessing for standard cleaned text 
y = df["emotion"]

# Inspect the results
display(
    df[["clean_text", "neg_text", "emotion"]].head() #syntax needs double brackets here to select multiple col from df 
)

print("Number of input examples:", len(X))
print("Number of labels:", len(y))


,clean_text,neg_text,emotion
0,feel humiliated,not_feel humiliated,sadness
1,feel hopeless damned hopeful care awake,feel hopeless damned hopeful care awake,sadness
2,m grab minute post feel greedy wrong,m grab minute post feel greedy wrong,anger
3,feel nostalgic fireplace know property,feel nostalgic fireplace know property,love
4,feel grouchy,feel grouchy,anger


Number of input examples: 16000
Number of labels: 16000


Train/ Test Split and Cross Validation

Divide the dataset innto 80% training, 20% testing with a fixed random seed 

will be used on standard cleaned and negation aware text 

5 fold stratified cross validation used on the training data to eval model performance 





In [66]:

# Split row indices, not the text itself
train_idx, test_idx = train_test_split( #scikit-learn function that divides data into separate groups 
    df.index.to_numpy(), #identifies the row of the df and converts them ito NumPy array 
    test_size=0.20, #reserve 20% for testing 
    random_state=42, #make random split reproducible 
    stratify=y #preserve the ratios / emotion proportions in each group 
)

# Primary model uses negation-aware text
X_train = df.loc[train_idx, "neg_text"] 
X_test = df.loc[test_idx, "neg_text"]

# Corresponding emotion labels
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

# Define five-fold stratified cross-validation aka cross validation splitter 
cv = StratifiedKFold( #variable created to store a cross validation configuration using a method of dividing training data into balanced folds for repeat eval 
    n_splits=5, #5 folds 
    shuffle=True, #randomize the order before assigning folds 
    random_state=42 #makes this experiement reproducible 
)

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))


Training examples: 12800
Testing examples: 3200


CountVectorizer and Multinomial Naive Bayes - Baseline established

CountVectorizer convert preprocessed text into num word count features

MNB learns statistical relationship between how many times word occurs and the emotion label

5 fold stratified cross validation on training set uses macro F1

Baseline is used as a reference point 



In [67]:

# Create the baseline model pipeline using the following:
#CountVectorizer: Converts text into numbers by counting words.
#Multinomial Naive Bayes: Learns which words are associated with each emotion.
#Macro F1: Measures how well the classifier performs across the emotion categories.
#Five-fold cross-validation: Repeats training and validation across different portions of the training data.

baseline_model = Pipeline([ #scikit-learn class that connects processing steps using pipeline to automate a sequence of operations 
    (
        "vectorizer",
        CountVectorizer(token_pattern=r"(?u)\b\w+\b") #creates a tool that turns sentences into numerical features, CountVectorizer builds a vocabulary and counts each word. #regex
    ),
    (
        "classifier",
        MultinomialNB(alpha=1.0) #created NB classifier, suitable for non negative text features using smoothing parameter, alpha, avoids zero probability 
    )
])

# Evaluate the model using five-fold cross-validation
baseline_scores = cross_val_score( #Train and evaluate the model across multiple folds
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro" #combines Precision and Recall 
)

# Display the results
print("Macro F1 scores:", baseline_scores)
print("Mean Macro F1:", baseline_scores.mean()) #avg score across folds
print("Standard deviation:", baseline_scores.std()) #how much scores vary


Macro F1 scores: [0.58581226 0.60805801 0.6058271  0.57914435 0.59196101]
Mean Macro F1: 0.5941605457566481
Standard deviation: 0.011218432829612017


Experiment 1 

Standard vs Negation Aware Preprocessing 

Does negation-aware preprocessing improve emotion classification performance?



In [68]:

# Use the same training rows for both experiments
#train_idx contains the row indices chosen for training in Cell 9.
X_train_clean = df.loc[train_idx, "clean_text"] #nee variable storing cleaning training sentences by locating the rows and cols by their labels in df using training row indicies
X_train_neg = df.loc[train_idx, "neg_text"]

# Evaluate standard preprocessing
clean_scores = cross_val_score( #takes NB pipeline, feeds in standard cleaned training sentences and emotions labels
    baseline_model,
    X_train_clean,
    y_train,
    cv=cv, #trains and evaluates 5 times using folds defined by cv
    scoring="f1_macro" #returns macro f1 score and stroes it in variable clean_scores 
)

# Evaluate negation-aware preprocessing
neg_scores = cross_val_score(  #takes NB pipeline, feeds in negated cleaned training sentences and emotions labels
    baseline_model,
    X_train_neg,
    y_train,
    cv=cv,
    scoring="f1_macro"
)

# Compare results
print("Standard preprocessing:", clean_scores.mean())
print("Negation-aware preprocessing:", neg_scores.mean())
print("Difference (neg - clean):", neg_scores.mean() - clean_scores.mean())


Standard preprocessing: 0.6082953856035564
Negation-aware preprocessing: 0.5941605457566481
Difference (neg - clean): -0.014134839846908331


Interpretation: Under the controlled experimental conditions, negation-aware preprocessing did not improve Multinomial Naive Bayes classification performance. This suggests that the additional negation handling may not provide a benefit for this model configuration.

Next steps: Compare CountVectorizer and TF-IDF, then evaluate Naive Bayes against Logistic Regression using consistent data splits and evaluation procedures.
Comparing feature extraction while keeping Classifier, Smoothing and Token Pattern the same

Does TF-IDF improve emotion classification compared with word-count features?



In [69]:

#CountVectorizer represents frequency. TF-IDF represents frequency adjusted by how common a word is across documents.


# Model A: CountVectorizer + Naive Bayes
count_model = Pipeline([
    ("vectorizer", CountVectorizer(token_pattern=r"(?u)\b\w+\b")),
    ("classifier", MultinomialNB(alpha=1.0))
])

# Model B: TF-IDF + Naive Bayes
tfidf_model = Pipeline([
    ("vectorizer", TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")),
    ("classifier", MultinomialNB(alpha=1.0))
])

# Evaluate both models on identical folds
count_scores = cross_val_score(
    count_model, X_train, y_train,
    cv=cv, scoring="f1_macro"
)

tfidf_scores = cross_val_score(
    tfidf_model, X_train, y_train,
    cv=cv, scoring="f1_macro"
)

# Compare results
print("CountVectorizer Macro F1:", count_scores.mean())
print("TF-IDF Macro F1:", tfidf_scores.mean())
print("Difference (TF-IDF - Count):",
      tfidf_scores.mean() - count_scores.mean())


CountVectorizer Macro F1: 0.5941605457566481
TF-IDF Macro F1: 0.41506475456201253
Difference (TF-IDF - Count): -0.1790957911946356


In [70]:
from sklearn.base import clone

balanced_model = clone(tfidf_model)

In [73]:
from sklearn.model_selection import cross_validate

balanced_cv_results = cross_validate(
    balanced_model,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro",
    return_estimator=True
)

Experiment #3:

Creeating a new cell to get standard pre processed text for the same training rows 

In [71]:
X_train_clean = df.loc[train_idx,"clean_text"]
print(len(X_train_clean))

#needs model, input text, correct answers and cross validation
print("Represents Macro F1 score from 5 cross validation fold: ")
standard_f1_scores = cross_val_score(tfidf_model, X_train_clean, y_train, cv=cv, scoring="f1_macro"
)

print(standard_f1_scores)
print(standard_f1_scores.mean())

12800
Represents Macro F1 score from 5 cross validation fold: 
[0.44035894 0.43012069 0.43036924 0.42173033 0.42869065]
0.4302539685078637


Mark's Experiment — Class Balancing
4.

In [72]:
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

tfidf_model.fit(X_train, y_train, classifier__sample_weight=sample_weights)
y_pred = tfidf_model.predict(X_test)

print(classification_report(
    y_test, y_pred, digits=3
))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

       anger      0.774     0.817     0.795       432
        fear      0.766     0.829     0.797       387
         joy      0.910     0.765     0.831      1072
        love      0.592     0.835     0.693       261
     sadness      0.908     0.794     0.847       933
    surprise      0.400     0.835     0.541       115

    accuracy                          0.797      3200
   macro avg      0.725     0.813     0.751      3200
weighted avg      0.829     0.797     0.805      3200

Accuracy: 0.7965625
Balanced accuracy: 0.8126260394326345
Macro F1: 0.7506904583448969
Confusion matrix:
[[353  16  15  13  19  16]
 [ 14 321   8   9   6  29]
 [ 42  29 820  94  41  46]
 [  4   2  25 218   7   5]
 [ 43  40  28  33 741  48]
 [  0  11   5   1   2  96]]


class-balanced cross-validation

In [94]:
balanced_f1_scores = []

for train_fold, val_fold in cv.split(X_train, y_train): 

    X_fold_train = X_train.iloc[train_fold] #training text
    X_fold_val = X_train.iloc[val_fold] #validation text
    y_fold_train = y_train.iloc[train_fold] #training emotion labels
    y_fold_val = y_train.iloc[val_fold] #validation emotion labels 

    equal_weights = compute_sample_weight(class_weight='balanced', y=y_fold_train)

    model = clone(balanced_model) #fresh, untrained model for the current fold.

    model.fit(X_fold_train,y_fold_train,classifier__sample_weight=equal_weights)

    predictions= model.predict(X_fold_val)

    score = f1_score(y_fold_val,predictions, average="macro")

    balanced_f1_scores.append(score)

print ("TF-IDF with class balance: ", balanced_f1_scores)
print("Average:  ", np.mean(balanced_f1_scores))
print("TF-IDF without class balance: ", balanced_cv_results["test_score"])


TF-IDF with class balance:  [0.7538781874782855, 0.7454348950808164, 0.7494550162301495, 0.7561619145604471, 0.753557755398389]
Average:   0.7516975537496176
TF-IDF without class balance:  [0.41998054 0.41455431 0.41354229 0.40846029 0.41878634]
